In [1]:
import os
import sys
import re

# =========================================================
# 1. ÉP VERSION PYSPARK (PHẢI CHẠY ĐẦU TIÊN TRƯỚC KHI IMPORT)
# =========================================================
modules_to_remove = [mod for mod in sys.modules if mod.startswith('pyspark') or mod.startswith('py4j')]
for mod in modules_to_remove: 
    del sys.modules[mod]

sys.path = [p for p in sys.path if "/usr/local/spark" not in p]
if "PYTHONPATH" in os.environ: 
    del os.environ["PYTHONPATH"]
    
conda_site_packages = "/opt/conda/lib/python3.13/site-packages"
if conda_site_packages not in sys.path: 
    sys.path.insert(0, conda_site_packages)
    
os.environ["SPARK_HOME"] = os.path.join(conda_site_packages, "pyspark")
os.environ["PYSPARK_PYTHON"] = "python3"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python3"


# =========================================================
# 2. BÂY GIỜ MỚI IMPORT PYSPARK VÀ KHỞI TẠO SESSION
# =========================================================
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, concat_ws, udf
from pyspark.sql.types import FloatType

spark = SparkSession.builder \
    .appName("VADER_Pipeline") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "dataNLPmining-lab") \
    .config("spark.hadoop.fs.s3a.secret.key", "dataNLPmining-lab") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.my_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.my_catalog.type", "hadoop") \
    .config("spark.sql.catalog.my_catalog.warehouse", "s3a://raw-financial-data/iceberg_warehouse") \
    .getOrCreate()


# =========================================================
# 3. VÁ LỖI THỜI GIAN HADOOP
# =========================================================
hadoop_conf = spark._jsc.hadoopConfiguration()
iterator = hadoop_conf.iterator()
while iterator.hasNext():
    entry = iterator.next()
    val = str(entry.getValue()).strip().lower()
    match = re.fullmatch(r"(\d+)([smhd])", val)
    if match:
        num, unit = int(match.group(1)), match.group(2)
        ms_val = num * 1000 if unit == 's' else num * 60000 if unit == 'm' else num * 3600000 if unit == 'h' else num * 86400000
        hadoop_conf.set(entry.getKey(), str(ms_val))

print("✅ Khởi tạo Spark và môi trường hoàn tất!")

✅ Khởi tạo Spark và môi trường hoàn tất!


In [3]:
#file parquet nằm trong iceberg-warehouse
df_lemmas = spark.read.table("my_catalog.processed_zone_finnhub.news_lemmas")
df_lemmas.show(5, truncate=False)

+---------+-------------------+-------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|id       |published_at       |title_lemmas                                                                                     |summary_lemmas                                                                                                                                                                                     

In [4]:
!pip install tensorflow


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.9/572.9 MB 5.3 MB/s  0:01:17m0:00:0100:03m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 4.5 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 7.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 8.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 7.5 MB/s  0:00:03m0:00:0100:01
  Attempting uninstall: h5py0m╸━━━━━━━━━━━━━━━━━━━━━  7/15 [ml_dtypes]]
    Found existing installation: h5py 3.16.0━━━━━━━━━━━━━━━━━━  7/15 [ml_dtypes]
    Uninstalling h5py-3.16.0:╺━━━━━━━━━━━━━━━━━━  8/15 [h5py]]
      Successfully uninstalled h5py-3.16.0m━━━━━━━━━━━━━━━━━━  8/15 [h5py]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15/15 [tensorflow]5 [tensorflow]


In [4]:
from pyspark.ml.feature import Word2Vec

# Giả định df_lemmas là bảng dữ liệu bạn đã load từ iceberg-warehouse
# df_lemmas = spark.read.table("my_catalog.processed_zone.news_lemmas")

# 1. Khởi tạo thuật toán Word2Vec
word2Vec = Word2Vec(
    vectorSize=100,        # Kích thước vector (thường từ 50 đến 300)
    minCount=3,            # Bỏ qua những từ xuất hiện dưới 3 lần trong toàn bộ tập dữ liệu
    windowSize=5,          # Số lượng từ ngữ cảnh xung quanh từ đang xét (trước 5 từ, sau 5 từ)
    inputCol="title_lemmas", 
    outputCol="w2v_features",
    maxIter=100             # Số vòng lặp huấn luyện
)

print("Đang huấn luyện Word2Vec. Quá trình này có thể mất vài phút tùy vào dữ liệu...")
model_w2v = word2Vec.fit(df_lemmas)

# 2. Áp dụng mô hình để chuyển đổi chữ thành vector
df_vectorized = model_w2v.transform(df_lemmas)

# Xem kết quả
df_vectorized.select("id", "title_lemmas", "w2v_features").show(5, truncate=50)

# 3. Thử nghiệm tìm từ đồng nghĩa (Tìm 5 từ gần nghĩa nhất với từ "microsoft")
try:
    synonyms = model_w2v.findSynonyms("microsoft", 5)
    print("\nCác từ đồng nghĩa với 'microsoft':")
    synonyms.show()
except Exception as e:
    print("Từ 'microsoft' chưa đủ tần suất để đưa vào từ điển.")
    
# 4. Lưu lại mô hình để dùng sau này (không phải train lại)
# model_w2v.save("hdfs://path/to/save/word2vec_model")

Đang huấn luyện Word2Vec. Quá trình này có thể mất vài phút tùy vào dữ liệu...
+---------+--------------------------------------------------+--------------------------------------------------+
|       id|                                      title_lemmas|                                      w2v_features|
+---------+--------------------------------------------------+--------------------------------------------------+
|140651233|[market, chatter, jpmorgan, google, firm, ask, ...|[-0.10651516846635126,0.12842557206749916,-0.08...|
|140651234|[uk, regulator, set, new, rule, google, search,...|[-0.08092019360305534,0.08295166658030616,0.214...|
|140650770|[google, cloud, model, garden, platform, exclus...|[-0.0966306288133968,0.2530526671219956,0.04661...|
|140650768|[social, medium, political, problem, time, go, ...|[0.1754957742856017,0.018889869962419783,0.0086...|
|140649980|[custom, ai, chip, come, nvidia, crown, company...|[-0.05432535087068875,0.19298556141438894,-0.10...|
+--------

In [18]:
try:
    synonyms = model_w2v.findSynonyms("high",15)
    print("\nCác từ đồng nghĩa với 'facebook':")
    synonyms.show()
except Exception as e:
    print("Từ 'microsoft' chưa đủ tần suất để đưa vào từ điển.")


Các từ đồng nghĩa với 'facebook':
+---------+-------------------+
|     word|         similarity|
+---------+-------------------+
|      low| 0.6534342169761658|
|   record| 0.5922585129737854|
|      hit| 0.5642147064208984|
|     rise|0.47185200452804565|
|    climb|0.44273701310157776|
|    touch|0.43309709429740906|
|     trot|0.43274298310279846|
|    level| 0.4256684482097626|
|     near| 0.4182579815387726|
|    yield| 0.4164634644985199|
|stabilize|0.41346704959869385|
|   falter| 0.4127184748649597|
|      run|0.41200894117355347|
|  quality| 0.4021088480949402|
|    notch| 0.3964647352695465|
+---------+-------------------+



In [7]:
# Xác định đường dẫn trên S3
model_path = "s3a://raw-financial-data/models/word2vec_news_model"

# Lưu mô hình
model_w2v.save(model_path)
print(f"✅ Mô hình đã được lưu tại: {model_path}")

✅ Mô hình đã được lưu tại: s3a://raw-financial-data/models/word2vec_news_model
